In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path

In [ ]:
BASE_DIR = Path.cwd().parent

CATEGORY = "test"
COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

# The master file containing all subfigures and question structures
ORIGINAL_STATE_FILE = BASE_DIR / f"submission_finetuning_{CATEGORY}_state.json"
FINAL_SUBMISSION_PATH = BASE_DIR / f"submission_finetuning_{CATEGORY}.json"

# The 4 specific state files containing your generated answers
STATE_FILES = [
    BASE_DIR / f"submission_finetuning_{CATEGORY}__factoid_state.json",
    BASE_DIR / f"submission_finetuning_{CATEGORY}__paragraph_state.json",
    BASE_DIR / f"submission_finetuning_{CATEGORY}__list_state.json",
    BASE_DIR / f"submission_finetuning_{CATEGORY}__yes_no_state.json",
]

In [ ]:
def clean_and_validate_output(raw_answer: str, expected_type: str) -> tuple[str, bool]:
    """Cleans the answer and checks if it conforms to the expected type.
    Returns: (cleaned_answer, is_valid_format)
    """
    cleaned = raw_answer.strip()

    if expected_type == "Yes/No":
        ans_lower = cleaned.lower()
        # Use regex word boundaries (\b) to avoid matching "eyes", "note", "nothing", etc.
        if re.search(r"\byes\b", ans_lower):
            cleaned = "Yes"
        elif re.search(r"\bno\b", ans_lower):
            cleaned = "No"

    elif expected_type == "List":
        # Smart Splitting
        # If the LLM used newlines to separate items, split by those first.
        if "\n" in cleaned:
            raw_elements = [re.sub(r"^\s*,\s*", "", e) for e in cleaned.split("\n")]
        else:
            raw_elements = re.split(r",\s*(?![^()]*\))", cleaned)

        cleaned_elements = []
        for item in raw_elements:
            item = item.strip()
            if not item:
                continue

            # Targeted Bullet/Number Removal
            item = re.sub(
                r"^\s*(?:[\*\•]|\-\s*|\-(?=[a-zA-Z])|\(?(?:[a-zA-Z]{1,2}|\d{1,2})\s*[.\):\-]+\s+)\s*",
                "",
                item,
            )

            item = re.sub(r"[.!?]+$", "", item)

            # Normalize internal whitespace
            item = re.sub(r"\s+", " ", item)

            # Filter out junk artifacts, trailing full stops, AND hanging commas
            item = item.strip(" -:.,")

            if item:
                cleaned_elements.append(item)

        # Join as a clean, comma-separated string
        cleaned = ", ".join(cleaned_elements)
        cleaned = re.sub(r"\s+", " ", cleaned).strip()

    elif expected_type == "Factoid":
        # Remove trailing punctuation often mistakenly added by LLMs to short terms
        cleaned = re.sub(r"[.!?]+$", "", cleaned)
        # Replace multiple spaces/newlines with single spaces for a clean paragraph
        cleaned = re.sub(r"\s+", " ", cleaned).strip()

    elif expected_type == "Paragraph":
        # Replace multiple spaces/newlines with single spaces for a clean paragraph
        cleaned = re.sub(r"\s+", " ", cleaned).strip()

    return cleaned, cleaned != raw_answer

In [ ]:
def build_validated_submission():
    # 1. Load all generated answers into a single lookup dictionary
    # Structure: answers_lookup[(sample_id, sub_fig, question_text)] = answer_text
    answers_lookup = {}
    seen_keys = set()

    print("Loading partial states...")
    for state_file in STATE_FILES:
        if not state_file.exists():
            print(f"  [!] Warning: {state_file.name} not found. Skipping.")
            continue

        with open(state_file) as f:
            partial_state = json.load(f)

        for sample_id, sub_figs in partial_state.items():
            for sub_fig, questions in sub_figs.items():
                # Handle both dict and list formats securely
                if isinstance(questions, dict):
                    items_to_process = questions.items()
                elif isinstance(questions, list):
                    items_to_process = [(q["question"], q["answer"]) for q in questions]
                else:
                    continue

                for q_text, ans_text in items_to_process:
                    unique_key = (sample_id, sub_fig, q_text)

                    if unique_key not in seen_keys:
                        seen_keys.add(unique_key)
                        answers_lookup[unique_key] = ans_text

    print(f"✅ Loaded {len(answers_lookup)} unique answers from state files.\n")

    # 2. Parse the ground truth directory to build the exact expected submission structure
    print("Scanning dataset directory to build guaranteed submission structure...")
    json_files = list(CASE_DIR.rglob("*.json"))

    submission = []
    missing_answers_count = 0
    invalid_format_count = 0
    expected_questions_count = 0

    # Dictionary to track missing counts by answer_type
    missing_per_type = defaultdict(int)

    for json_file in json_files:
        fullpath = str(json_file)
        if (
            "content.json" in json_file.name
            or "images" not in fullpath
            or ".vscode" in fullpath
        ):
            continue

        with open(json_file) as f:
            data = json.load(f)

        sample_id = data["sample_id"]
        vqa_data = data.get("vqa", {})

        sample_output_vqa = {}

        for sub_fig, q_list in vqa_data.items():
            sample_output_vqa[sub_fig] = []

            for q_obj in q_list:
                expected_questions_count += 1
                q_text = q_obj.get("question") or q_obj.get("questions")
                ans_type = q_obj.get("answer_type", "Unknown")

                # Retrieve the answer from our lookup, fallback to empty string if missing
                unique_key = (sample_id, sub_fig, q_text)
                ans_text = answers_lookup.get(unique_key, "")

                # Clean the answer if it exists
                if ans_text:
                    ans_text, is_valid = clean_and_validate_output(ans_text, ans_type)
                    if not is_valid:
                        invalid_format_count += 1

                if not ans_text:
                    missing_answers_count += 1
                    missing_per_type[ans_type] += 1
                    print(
                        f"  [!] Missing/Empty [{ans_type}] Answer for {sample_id} | {sub_fig} | Q: {q_text[:30]}..."
                    )

                # Append strictly exactly what the evaluator expects
                sample_output_vqa[sub_fig].append(
                    {
                        "question": q_text,
                        "question_type": q_obj.get("question_type", ""),
                        "answer_type": ans_type,
                        "answer": ans_text,
                    }
                )

        submission.append({"sample_id": sample_id, "vqa": sample_output_vqa})

    # 3. Save the final submission
    with open(FINAL_SUBMISSION_PATH, "w") as f:
        json.dump(submission, f, indent=2)

    # 4. Print detailed summary
    print(f"📊 Expected Questions: {expected_questions_count}")
    print(
        f"📊 Answers Found (after cleaning): {expected_questions_count - missing_answers_count}"
    )
    print(f"📊 Answers Flagged as Invalid Format: {invalid_format_count}")
    print(f"📊 Answers Missing Total: {missing_answers_count}")

    if missing_answers_count > 0:
        print("   Breakdown of missing answers by type:")
        for ans_type, count in missing_per_type.items():
            print(f"    - {ans_type}: {count}")

    print(
        f"\n💾 Saved strictly validated submission ({len(submission)} samples) to {FINAL_SUBMISSION_PATH.name}"
    )

In [ ]:
build_validated_submission()